In [54]:
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# from xgboost import XGBRegressor
# from lightgbm import LGBMRegressor

# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import (
#     mean_absolute_error,
#     mean_squared_error,
#     r2_score,
# )

# # -------------------------------------------------------------------
# # 0. Paths and global config
# # -------------------------------------------------------------------

# DATA_PATH = "data.csv"

# # Folder for saving plots
# PLOTS_DIR = "plots"
# os.makedirs(PLOTS_DIR, exist_ok=True)

# # Date boundaries for splits
# TRAIN_END = pd.Timestamp("2016-12-31")
# VAL_END = pd.Timestamp("2021-12-31")

# # Spread horizon windows (days) – consistent with naming
# SPREAD_RET_WINDOWS = [30, 60, 180, 360]
# SPREAD_VOL_WINDOWS = [30, 60, 180, 360]

# # Macro diff horizons (days) with semantic labels
# MACRO_DIFF_HORIZONS = [
#     (7, "week"),
#     (30, "month"),
#     (180, "half_y"),
#     (360, "year"),
# ]


# # -------------------------------------------------------------------
# # 1. Feature engineering
# # -------------------------------------------------------------------

# def build_features(df: pd.DataFrame):
#     """
#     Perform all feature engineering on a copy of df:
#     - Clean column names
#     - Ensure 'date' is datetime and sorted
#     - Add spread features (returns & volatility)
#     - Add macro diff features (week/month/half_y/year)

#     Returns:
#         df_fe    : engineered dataframe (with 'spread' and no NaNs)
#         num_cols : list of numeric feature column names
#         cat_cols : list of categorical feature column names (empty for now)
#     """
#     df = df.copy()

#     # Strip whitespace that may appear in column names
#     df.columns = df.columns.str.strip()

#     # Ensure we have a 'date' column
#     if "date" not in df.columns:
#         # Assume first column is date if not explicitly named
#         df.rename(columns={df.columns[0]: "date"}, inplace=True)

#     # Explicit datetime handling
#     df["date"] = pd.to_datetime(df["date"], errors="raise")

#     # Sort chronologically
#     df = df.sort_values("date").reset_index(drop=True)

#     # Base macro features (from original notebook)
#     base_features = [
#         "^VIX",
#         "^GSPC",
#         "CCC_10Y_Spread",
#         "Unemployment",
#         "Inflation",
#         "GDP_Growth",
#         "Corporate_Leverage",
#         "YieldCurveSlope",
#         "IndPro_Growth",
#         "CP_Growth",
#     ]

#     # Sanity check: all base features must exist
#     missing = [f for f in base_features if f not in df.columns]
#     if missing:
#         raise ValueError(f"Missing base features in input data: {missing}")

#     # Target spread series
#     df["spread"] = df["CCC_10Y_Spread"]

#     # Spread returns over various horizons (using levels)
#     for w in SPREAD_RET_WINDOWS:
#         df[f"spread_ret_{w}"] = df["spread"].diff(w)

#     # Spread volatility over various horizons (std of daily changes)
#     for w in SPREAD_VOL_WINDOWS:
#         df[f"spread_vol_{w}"] = df["spread"].diff().rolling(w).std()

#     # First differences of macro features over multiple horizons
#     for col in base_features:
#         for lag, label in MACRO_DIFF_HORIZONS:
#             df[f"{col}_{label}"] = df[col].diff(lag)

#     # Drop rows with NaNs created by diff/rolling
#     df = df.dropna().reset_index(drop=True)

#     # Construct numeric feature list
#     spread_ret_cols = [f"spread_ret_{w}" for w in SPREAD_RET_WINDOWS]
#     spread_vol_cols = [f"spread_vol_{w}" for w in SPREAD_VOL_WINDOWS]

#     macro_diff_cols = []
#     for col in base_features:
#         for _, label in MACRO_DIFF_HORIZONS:
#             macro_diff_cols.append(f"{col}_{label}")

#     feature_cols_num = base_features + spread_ret_cols + spread_vol_cols + macro_diff_cols

#     # Placeholder for categoricals if you add them later
#     feature_cols_cat = []  # e.g., ['rating', 'sector']

#     return df, feature_cols_num, feature_cols_cat


# # -------------------------------------------------------------------
# # 2. Metrics with directional accuracy
# # -------------------------------------------------------------------

# def metrics_with_direction(y_true, y_pred):
#     """
#     Compute MAE, RMSE, R^2, MAPE, and directional accuracy
#     (sign agreement of first differences).
#     """
#     y_true = np.asarray(y_true)
#     y_pred = np.asarray(y_pred)

#     mae = mean_absolute_error(y_true, y_pred)
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     r2 = r2_score(y_true, y_pred)

#     # MAPE on levels
#     denom = np.where(y_true == 0, 1e-8, y_true)
#     mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100

#     # Directional Accuracy based on changes in level
#     true_dir = np.sign(y_true[1:] - y_true[:-1])
#     pred_dir = np.sign(y_pred[1:] - y_pred[:-1])
#     da = np.mean(true_dir == pred_dir)

#     return mae, rmse, r2, mape, da


# def print_metrics_block(label, y_true, y_pred):
#     mae, rmse, r2, mape, da = metrics_with_direction(y_true, y_pred)
#     print(f"=== {label} ===")
#     print(f"MAE : {mae:.4f}")
#     print(f"RMSE: {rmse:.6f}")
#     print(f"R²  : {r2:.4f}")
#     print(f"MAPE: {mape:.2f}%")
#     print(f"DA  : {da:.4f}")
#     print()


# # -------------------------------------------------------------------
# # 3. Plotting utilities (saving to disk)
# # -------------------------------------------------------------------

# def plot_time_series(dates, y_true, y_pred_dict, title, filename):
#     """
#     Save a time-series plot of actual vs predicted spreads.
#     """
#     plt.figure(figsize=(12, 6))
#     plt.plot(dates, y_true, label="Actual", linewidth=2)
#     for name, y_pred in y_pred_dict.items():
#         plt.plot(dates, y_pred, label=f"{name} pred", alpha=0.8)

#     plt.xlabel("Date")
#     plt.ylabel("Spread")
#     plt.title(title)
#     plt.legend()
#     plt.tight_layout()

#     save_path = os.path.join(PLOTS_DIR, filename)
#     plt.savefig(save_path, dpi=300)
#     plt.close()
#     print(f"Saved plot: {save_path}")


# def plot_scatter(y_true, y_pred, model_name, split_name, filename):
#     """
#     Save scatter plot of predicted vs actual values.
#     """
#     plt.figure(figsize=(7, 7))
#     plt.scatter(y_true, y_pred, alpha=0.5)

#     min_val = min(y_true.min(), y_pred.min())
#     max_val = max(y_true.max(), y_pred.max())
#     plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

#     plt.xlabel("Actual")
#     plt.ylabel("Predicted")
#     plt.title(f"{model_name} – {split_name} actual vs predicted")
#     plt.tight_layout()

#     save_path = os.path.join(PLOTS_DIR, filename)
#     plt.savefig(save_path, dpi=300)
#     plt.close()
#     print(f"Saved plot: {save_path}")


# def plot_feature_importances(tree_model, feature_names, model_name, filename, top_n=20):
#     """
#     Save bar plot of top_n feature importances.
#     """
#     importances = getattr(tree_model, "feature_importances_", None)
#     if importances is None:
#         print(f"{model_name}: no feature_importances_ attribute; skipping plot.")
#         return

#     importances = np.asarray(importances)

#     # Align lengths defensively
#     n = min(len(importances), len(feature_names))
#     importances = importances[:n]
#     feature_names = feature_names[:n]

#     # Select top_n
#     if top_n is not None and top_n < n:
#         idx = np.argsort(importances)[-top_n:]
#     else:
#         idx = np.argsort(importances)

#     sorted_importances = importances[idx]
#     sorted_names = [feature_names[i] for i in idx]

#     plt.figure(figsize=(10, 6))
#     plt.barh(range(len(sorted_importances)), sorted_importances)
#     plt.yticks(range(len(sorted_importances)), sorted_names)
#     plt.xlabel("Importance")
#     plt.title(f"{model_name} – top {len(sorted_importances)} feature importances")
#     plt.tight_layout()

#     save_path = os.path.join(PLOTS_DIR, filename)
#     plt.savefig(save_path, dpi=300)
#     plt.close()
#     print(f"Saved plot: {save_path}")


# # -------------------------------------------------------------------
# # 4. Main training / evaluation routine
# # -------------------------------------------------------------------

# def main():
#     # 4.1 Load data
#     df_raw = pd.read_csv(DATA_PATH)

#     # 4.2 Feature engineering
#     df, feature_cols_num, feature_cols_cat = build_features(df_raw)

#     # 4.3 Train/validation/test split
#     train = df[df["date"] <= TRAIN_END]
#     val = df[(df["date"] > TRAIN_END) & (df["date"] <= VAL_END)]
#     test = df[df["date"] > VAL_END]

#     # Features and target
#     X_train = train[feature_cols_num]
#     X_val = val[feature_cols_num]
#     X_test = test[feature_cols_num]

#     y_train = train["spread"].values
#     y_val = val["spread"].values
#     y_test = test["spread"].values

#     # ----------------------------------------------------------------
#     # 4.4 Preprocessors (ColumnTransformers) and Pipelines
#     # ----------------------------------------------------------------

#     # For XGBoost: scale numeric features
#     preprocessor_xgb = ColumnTransformer(
#         transformers=[
#             ("num", StandardScaler(), feature_cols_num),
#             ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
#         ],
#         remainder="drop",
#     )

#     xgb_model = XGBRegressor(
#         n_estimators=3000,
#         max_depth=20,
#         learning_rate=0.01,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         min_child_weight=5,
#         reg_lambda=2.0,
#         reg_alpha=1.0,
#         objective="reg:squarederror",
#         random_state=42,
#     )

#     xgb_pipeline = Pipeline(
#         steps=[
#             ("preprocess", preprocessor_xgb),
#             ("model", xgb_model),
#         ]
#     )

#     # For LightGBM: do not scale numeric features (pass-through)
#     preprocessor_lgb = ColumnTransformer(
#         transformers=[
#             ("num", "passthrough", feature_cols_num),
#             ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
#         ],
#         remainder="drop",
#     )

#     lgbm_model = LGBMRegressor(
#         n_estimators=3000,
#         learning_rate=0.01,
#         num_leaves=31,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=2.0,
#         reg_alpha=1.0,
#         random_state=42,
#         n_jobs=-1,
#     )

#     lgbm_pipeline = Pipeline(
#         steps=[
#             ("preprocess", preprocessor_lgb),
#             ("model", lgbm_model),
#         ]
#     )

#     # ----------------------------------------------------------------
#     # 4.5 Fit models
#     # ----------------------------------------------------------------

#     print("Fitting XGBoost...")
#     xgb_pipeline.fit(X_train, y_train)

#     print("Fitting LightGBM...")
#     lgbm_pipeline.fit(X_train, y_train)

#     # ----------------------------------------------------------------
#     # 4.6 Predictions and metrics
#     # ----------------------------------------------------------------

#     # XGBoost
#     y_pred_train_x = xgb_pipeline.predict(X_train)
#     y_pred_val_x = xgb_pipeline.predict(X_val)
#     y_pred_test_x = xgb_pipeline.predict(X_test)

#     print("\nXGBoost metrics:")
#     print_metrics_block("TRAIN (level + direction)", y_train, y_pred_train_x)
#     print_metrics_block("VAL   (level + direction)", y_val, y_pred_val_x)
#     print_metrics_block("TEST  (level + direction)", y_test, y_pred_test_x)

#     # LightGBM
#     y_pred_train_l = lgbm_pipeline.predict(X_train)
#     y_pred_val_l = lgbm_pipeline.predict(X_val)
#     y_pred_test_l = lgbm_pipeline.predict(X_test)

#     print("LightGBM metrics:")
#     print_metrics_block("TRAIN (level + direction)", y_train, y_pred_train_l)
#     print_metrics_block("VAL   (level + direction)", y_val, y_pred_val_l)
#     print_metrics_block("TEST  (level + direction)", y_test, y_pred_test_l)

#     # ----------------------------------------------------------------
#     # 4.7 Plots (saved to disk)
#     # ----------------------------------------------------------------

#     # Time series comparison on validation
#     plot_time_series(
#         dates=val["date"].values,
#         y_true=y_val,
#         y_pred_dict={
#             "XGBoost": y_pred_val_x,
#             "LightGBM": y_pred_val_l,
#         },
#         title="Validation: actual vs predicted CCC_10Y_Spread",
#         filename="val_timeseries.png",
#     )

#     # Scatter plots (validation)
#     plot_scatter(
#         y_val,
#         y_pred_val_x,
#         model_name="XGBoost",
#         split_name="VAL",
#         filename="xgb_scatter_val.png",
#     )
#     plot_scatter(
#         y_val,
#         y_pred_val_l,
#         model_name="LightGBM",
#         split_name="VAL",
#         filename="lgb_scatter_val.png",
#     )

#     # Feature importance plots
#     xgb_est = xgb_pipeline.named_steps["model"]
#     lgbm_est = lgbm_pipeline.named_steps["model"]

#     plot_feature_importances(
#         xgb_est,
#         feature_cols_num,
#         model_name="XGBoost",
#         filename="xgb_feature_importances.png",
#         top_n=20,
#     )
#     plot_feature_importances(
#         lgbm_est,
#         feature_cols_num,
#         model_name="LightGBM",
#         filename="lgb_feature_importances.png",
#         top_n=20,
#     )


# if __name__ == "__main__":
#     main()


# Import

In [55]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# -------------------------------------------------------------------
# 0. Paths and global config
# -------------------------------------------------------------------

DATA_PATH = "data.csv"

# Folder for saving plots
PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# Date boundaries for splits
TRAIN_END = pd.Timestamp("2016-12-31")
VAL_END = pd.Timestamp("2021-12-31")

# Spread horizon windows (days) – consistent with naming
SPREAD_RET_WINDOWS = [30, 60, 180, 360]
SPREAD_VOL_WINDOWS = [30, 60, 180, 360]

# Macro diff horizons (days) with semantic labels
MACRO_DIFF_HORIZONS = [
    (7, "week"),
    (30, "month"),
    (180, "half_y"),
    (360, "year"),
]

df_raw = pd.read_csv(DATA_PATH)



# Feature Engineering

In [56]:
def build_features(df: pd.DataFrame):
    """
    Perform all feature engineering on a copy of df:
    - Clean column names
    - Ensure 'date' is datetime and sorted
    - Add spread features (returns & volatility)
    - Add macro diff features (week/month/half_y/year)

    Returns:
        df_fe    : engineered dataframe (with 'spread' and no NaNs)
        num_cols : list of numeric feature column names
        cat_cols : list of categorical feature column names (empty for now)
    """
    df = df.copy()

    # Strip whitespace that may appear in column names
    df.columns = df.columns.str.strip()

    # Ensure we have a 'date' column
    if "date" not in df.columns:
        # Assume first column is date if not explicitly named
        df.rename(columns={df.columns[0]: "date"}, inplace=True)

    # Explicit datetime handling
    df["date"] = pd.to_datetime(df["date"], errors="raise")

    # Sort chronologically
    df = df.sort_values("date").reset_index(drop=True)

    # Base macro features (from original notebook)
    base_features = [
        "^VIX",
        "^GSPC",
        "CCC_10Y_Spread",
        "Unemployment",
        "Inflation",
        "GDP_Growth",
        "Corporate_Leverage",
        "YieldCurveSlope",
        "IndPro_Growth",
        "CP_Growth",
    ]

    # Sanity check: all base features must exist
    missing = [f for f in base_features if f not in df.columns]
    if missing:
        raise ValueError(f"Missing base features in input data: {missing}")

    # Target spread series
    df["spread"] = df["CCC_10Y_Spread"]

    # Spread returns over various horizons (using levels)
    for w in SPREAD_RET_WINDOWS:
        df[f"spread_ret_{w}"] = df["spread"].diff(w)

    # Spread volatility over various horizons (std of daily changes)
    for w in SPREAD_VOL_WINDOWS:
        df[f"spread_vol_{w}"] = df["spread"].diff().rolling(w).std()

    # First differences of macro features over multiple horizons
    for col in base_features:
        for lag, label in MACRO_DIFF_HORIZONS:
            df[f"{col}_{label}"] = df[col].diff(lag)

    # Drop rows with NaNs created by diff/rolling
    df = df.dropna().reset_index(drop=True)

    # Construct numeric feature list
    spread_ret_cols = [f"spread_ret_{w}" for w in SPREAD_RET_WINDOWS]
    spread_vol_cols = [f"spread_vol_{w}" for w in SPREAD_VOL_WINDOWS]

    macro_diff_cols = []
    for col in base_features:
        for _, label in MACRO_DIFF_HORIZONS:
            macro_diff_cols.append(f"{col}_{label}")

    feature_cols_num = base_features + spread_ret_cols + spread_vol_cols + macro_diff_cols

    # Placeholder for categoricals if you add them later
    feature_cols_cat = []  # e.g., ['rating', 'sector']

    return df, feature_cols_num, feature_cols_cat

# Metrics

In [57]:
def metrics_with_direction(y_true, y_pred):
    """
    Compute MAE, RMSE, R^2, MAPE, and directional accuracy
    (sign agreement of first differences).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    # MAPE on levels
    denom = np.where(y_true == 0, 1e-8, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100

    # Directional Accuracy based on changes in level
    true_dir = np.sign(y_true[1:] - y_true[:-1])
    pred_dir = np.sign(y_pred[1:] - y_pred[:-1])
    da = np.mean(true_dir == pred_dir)

    return mae, rmse, r2, mape, da


def print_metrics_block(label, y_true, y_pred):
    mae, rmse, r2, mape, da = metrics_with_direction(y_true, y_pred)
    print(f"=== {label} ===")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R²  : {r2:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"DA  : {da:.4f}")
    print()


# Plotting

In [58]:
def plot_time_series(dates, y_true, y_pred_dict, title, filename):
    """
    Save a time-series plot of actual vs predicted spreads.
    """
    plt.figure(figsize=(12, 6))
    plt.plot(dates, y_true, label="Actual", linewidth=2)
    for name, y_pred in y_pred_dict.items():
        plt.plot(dates, y_pred, label=f"{name} pred", alpha=0.8)

    plt.xlabel("Date")
    plt.ylabel("Spread")
    plt.title(title)
    plt.legend()
    plt.tight_layout()

    save_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved plot: {save_path}")


def plot_scatter(y_true, y_pred, model_name, split_name, filename):
    """
    Save scatter plot of predicted vs actual values.
    """
    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, alpha=0.5)

    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{model_name} – {split_name} actual vs predicted")
    plt.tight_layout()

    save_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved plot: {save_path}")


def plot_feature_importances(tree_model, feature_names, model_name, filename, top_n=20):
    """
    Save bar plot of top_n feature importances.
    """
    importances = getattr(tree_model, "feature_importances_", None)
    if importances is None:
        print(f"{model_name}: no feature_importances_ attribute; skipping plot.")
        return

    importances = np.asarray(importances)

    # Align lengths defensively
    n = min(len(importances), len(feature_names))
    importances = importances[:n]
    feature_names = feature_names[:n]

    # Select top_n
    if top_n is not None and top_n < n:
        idx = np.argsort(importances)[-top_n:]
    else:
        idx = np.argsort(importances)

    sorted_importances = importances[idx]
    sorted_names = [feature_names[i] for i in idx]

    plt.figure(figsize=(10, 6))
    plt.barh(range(len(sorted_importances)), sorted_importances)
    plt.yticks(range(len(sorted_importances)), sorted_names)
    plt.xlabel("Importance")
    plt.title(f"{model_name} – top {len(sorted_importances)} feature importances")
    plt.tight_layout()

    save_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved plot: {save_path}")


In [59]:
df, feature_cols_num, feature_cols_cat = build_features(df_raw)

# 4.3 Train/validation/test split
train = df[df["date"] <= TRAIN_END]
val = df[(df["date"] > TRAIN_END) & (df["date"] <= VAL_END)]
test = df[df["date"] > VAL_END]

# Features and target
X_train = train[feature_cols_num]
X_val = val[feature_cols_num]
X_test = test[feature_cols_num]

y_train = train["spread"].values
y_val = val["spread"].values
y_test = test["spread"].values

# Main

In [ ]:



# ----------------------------------------------------------------
# 4.4 Preprocessors (ColumnTransformers) and Pipelines
# ----------------------------------------------------------------

# For XGBoost: scale numeric features
preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), feature_cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
    ],
    remainder="drop",
)

xgb_model = XGBRegressor(
    n_estimators=3000,
    max_depth=20,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=2.0,
    reg_alpha=1.0,
    objective="reg:squarederror",
    random_state=42,
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_xgb),
        ("model", xgb_model),
    ]
)

# For LightGBM: do not scale numeric features (pass-through)
preprocessor_lgb = ColumnTransformer(
    transformers=[
        ("num", "passthrough", feature_cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
    ],
    remainder="drop",
)

lgbm_model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=1.0,
    random_state=42,
    n_jobs=-1,
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor_lgb),
        ("model", lgbm_model),
    ]
)

# ----------------------------------------------------------------
# 4.5 Fit models
# ----------------------------------------------------------------

# print("Fitting XGBoost...")
# xgb_pipeline.fit(X_train, y_train)

print("Fitting LightGBM...")
lgbm_pipeline.fit(X_train, y_train)

# ----------------------------------------------------------------
# 4.6 Predictions and metrics
# ----------------------------------------------------------------

# XGBoost
y_pred_train_x = xgb_pipeline.predict(X_train)
y_pred_val_x = xgb_pipeline.predict(X_val)
y_pred_test_x = xgb_pipeline.predict(X_test)

print("\nXGBoost metrics:")
print_metrics_block("TRAIN (level + direction)", y_train, y_pred_train_x)
print_metrics_block("VAL   (level + direction)", y_val, y_pred_val_x)
print_metrics_block("TEST  (level + direction)", y_test, y_pred_test_x)

# LightGBM
y_pred_train_l = lgbm_pipeline.predict(X_train)
y_pred_val_l = lgbm_pipeline.predict(X_val)
y_pred_test_l = lgbm_pipeline.predict(X_test)

print("LightGBM metrics:")
print_metrics_block("TRAIN (level + direction)", y_train, y_pred_train_l)
print_metrics_block("VAL   (level + direction)", y_val, y_pred_val_l)
print_metrics_block("TEST  (level + direction)", y_test, y_pred_test_l)

# ----------------------------------------------------------------
# 4.7 Plots (saved to disk)
# ----------------------------------------------------------------

# Time series comparison on validation
plot_time_series(
    dates=val["date"].values,
    y_true=y_val,
    y_pred_dict={
        "XGBoost": y_pred_val_x,
        "LightGBM": y_pred_val_l,
    },
    title="Validation: actual vs predicted CCC_10Y_Spread",
    filename="val_timeseries.png",
)

# Scatter plots (validation)
plot_scatter(
    y_val,
    y_pred_val_x,
    model_name="XGBoost",
    split_name="VAL",
    filename="xgb_scatter_val.png",
)
plot_scatter(
    y_val,
    y_pred_val_l,
    model_name="LightGBM",
    split_name="VAL",
    filename="lgb_scatter_val.png",
)

# Feature importance plots
xgb_est = xgb_pipeline.named_steps["model"]
lgbm_est = lgbm_pipeline.named_steps["model"]

plot_feature_importances(
    xgb_est,
    feature_cols_num,
    model_name="XGBoost",
    filename="xgb_feature_importances.png",
    top_n=20,
)
plot_feature_importances(
    lgbm_est,
    feature_cols_num,
    model_name="LightGBM",
    filename="lgb_feature_importances.png",
    top_n=20,
)


Fitting LightGBM...


ValueError: Pipeline.fit does not accept the force_col_wise parameter. You can pass parameters to specific steps of your pipeline using the stepname__parameter format, e.g. `Pipeline.fit(X, y, logisticregression__sample_weight=sample_weight)`.